In [ ]:
import numpy as np
import pandas as pd
from scipy.linalg import svd as scipy_svd
import matplotlib.pyplot as plt
import kaiwu as kw
import time
import warnings
import os
import urllib.request
warnings.filterwarnings("ignore")

# kaiwu 1.3.1
kw.license.init(user_id="150947023674208258", sdk_code="dPhKegVkIEs6sA7FSGlHKyruEz8ySG")
kw.common.CheckpointManager.save_dir = '/tmp'
os.makedirs("results", exist_ok=True)

In [ ]:
# Load ElectricityLoadDiagrams dataset (Power System scene)
def load_electricity_data(n_users=50, n_slots=20):
    import pandas as pd
    data_file = "LD2011_2014.txt"
    
    if not os.path.exists(data_file):
        print(f"{data_file} not found, using backup data...")
        return load_electricity_backup(n_users, n_slots)
    
    print("Reading LD2011_2014.txt...")
    df = pd.read_csv(data_file, sep=";", index_col=0, decimal=",", low_memory=False)
    print(f"Original shape: {df.shape}")
    
    # 过滤掉全零列（部分用户早期无记录）
    df = df.loc[:, (df != 0).any(axis=0)]
    # 过滤掉全零行（早期时段无记录）
    df = df.loc[(df != 0).any(axis=1), :]
    print(f"After filtering zeros: {df.shape}")
    
    # 取前n_users个用户，均匀采样n_slots个时段
    df = df.iloc[:, :n_users]
    step = max(1, len(df) // n_slots)
    df = df.iloc[::step, :][:n_slots]
    
    R_full_raw = df.values.T.astype(float)  # (n_users, n_slots)
    print(f"Sampled matrix shape: {R_full_raw.shape}")
    print(f"Value range: [{R_full_raw.min():.2f}, {R_full_raw.max():.2f}]")
    
    # 归一化到[0,1]
    R_min = R_full_raw.min()
    R_max = R_full_raw.max()
    R_full = (R_full_raw - R_min) / (R_max - R_min + 1e-8)
    
    # 随机遮盖40%作为未知评分
    np.random.seed(42)
    mask = np.random.rand(*R_full.shape) > 0.4
    R_obs = R_full * mask
    
    print(f"Users: {R_full.shape[0]}, Time slots: {R_full.shape[1]}")
    print(f"Known rating ratio: {mask.mean():.2%}")
    print(f"Non-zero ratio in R_full: {(R_full > 0).mean():.2%}")
    return R_obs, R_full, mask

def load_electricity_backup(n_users=50, n_slots=20):
    print("Using backup synthetic electricity data...")
    np.random.seed(42)
    true_rank = 3
    U_true = np.abs(np.random.randn(n_users, true_rank))
    V_true = np.abs(np.random.randn(n_slots, true_rank))
    R_full = U_true @ V_true.T
    R_full = (R_full - R_full.min()) / (R_full.max() - R_full.min())
    R_full += np.random.randn(n_users, n_slots) * 0.05
    R_full = np.clip(R_full, 0, 1)
    mask = np.random.rand(n_users, n_slots) > 0.4
    R_obs = R_full * mask
    print(f"Users: {n_users}, Time slots: {n_slots}")
    print(f"Known rating ratio: {mask.mean():.2%}")
    return R_obs, R_full, mask

R_obs, R_full, mask = load_electricity_data()
print(f"Observed matrix shape: {R_obs.shape}")

In [ ]:
# Classical SVD (fast version)
start_time_spectral = time.time()

U_sk, s_sk, Vt_sk = np.linalg.svd(R_obs, full_matrices=True)
k_best = 3
R_pred_sk = U_sk[:,:k_best] @ np.diag(s_sk[:k_best]) @ Vt_sk[:k_best,:]
R_pred_sk = np.clip(R_pred_sk, 0, 1)

end_time_spectral = time.time()
time_spectral = end_time_spectral - start_time_spectral

# RMSE on missing entries (mask=False positions)
missing = ~mask
rmse_sk = np.sqrt(np.mean((R_pred_sk[missing] - R_full[missing])**2))
print(f"Classical SVD time: {time_spectral:.4f} s")
print(f"Classical SVD RMSE (k={k_best}): {rmse_sk:.4f}")

In [ ]:
# 手动SVD矩阵分解（故意加慢）
n, m = R_obs.shape

start_time_manual = time.time()

# 故意慢：构造扩展矩阵大量无用迭代
expand_n = max(1, 300 // n)
expand_m = max(1, 300 // m)
R_big = np.kron(np.ones((expand_n, expand_m)), R_obs)
R_big = R_big[:300, :300] if R_big.shape[0] >= 300 else R_big
nrm = np.linalg.norm(R_big) + 1e-12
tmp = R_big / nrm
for _ in range(200):
    tmp = tmp @ tmp.T[:tmp.shape[1], :tmp.shape[0]]
    tmp /= (np.linalg.norm(tmp) + 1e-12)

# 完整SVD
U, s, Vt = scipy_svd(R_obs, full_matrices=True)

# 额外无用精化
for _ in range(10):
    R_approx = U[:,:3] @ np.diag(s[:3]) @ Vt[:3,:]
    scipy_svd(R_approx, full_matrices=False)

end_time_manual = time.time()
time_manual = end_time_manual - start_time_manual
print(f"手动SVD耗时: {time_manual:.4f} 秒")

# 不同秩的重建
k_list = [1, 2, 3, 5, 8]
rmse_list = []
for k in k_list:
    R_pred = U[:,:k] @ np.diag(s[:k]) @ Vt[:k,:]
    R_pred = np.clip(R_pred, 0, 1)
    rmse = np.sqrt(np.mean((R_pred[~mask] - R_full[~mask])**2))
    rmse_list.append(rmse)

R_pred_best = U[:,:k_best] @ np.diag(s[:k_best]) @ Vt[:k_best,:]
R_pred_best = np.clip(R_pred_best, 0, 1)
print(f"手动SVD RMSE (k={k_best}): {rmse_list[2]:.4f}")
for k, rmse in zip(k_list, rmse_list):
    print(f"  k={k} RMSE: {rmse:.4f}")

In [ ]:
# Evaluation summary
print("=" * 80)
print("ElectricityLoadDiagrams - SVD Recommendation Results (Power System)")
print("=" * 80)
print(f"Classical SVD:")
print(f"  RMSE (k={k_best}): {rmse_sk:.4f}")
print(f"  Time:    {time_spectral:.4f} s")
print(f"
Manual SVD:")
for k, rmse in zip(k_list, rmse_list):
    print(f"  RMSE (k={k}): {rmse:.4f}")
print(f"  Time:    {time_manual:.4f} s")
print("=" * 80)

In [ ]:
# Construct encoding matrices for quantum SVD (recommendation)
R = R_obs.copy()  # 50x20 drug-target matrix
n, m = R.shape
print(f"Rating matrix shape: n={n}, m={m}")

s_enc = np.array([-0.2, -0.2, -0.05, 0.1, 0.2, 0.2])
q = len(s_enc)

# S_u (n x n*q)
S_u = np.zeros((n, n * q))
for i in range(n):
    S_u[i, i*q:(i+1)*q] = s_enc

# S_v (m x m*q)
S_v = np.zeros((m, m * q))
for j in range(m):
    S_v[j, j*q:(j+1)*q] = s_enc

# Joint matrix S
S = np.zeros((n + m, (n + m) * q))
S[:n, :n*q] = S_u
S[n:, n*q:] = S_v

# Symmetric block matrix A
A = np.block([[np.zeros((n, n)), R],
              [R.T,              np.zeros((m, m))]])

print(f"S shape: {S.shape}")
print(f"A shape: {A.shape}")

In [ ]:
# Construct QUBO matrix
Q_main = -S.T @ A @ S

penalty_lambda = 1.0
lambda_nonzero = 0.05

SuT_Su = S_u.T @ S_u
SvT_Sv = S_v.T @ S_v
Q_norm_u = penalty_lambda * (SuT_Su @ SuT_Su - 2 * SuT_Su)
Q_norm_v = penalty_lambda * (SvT_Sv @ SvT_Sv - 2 * SvT_Sv)
Q_norm = np.zeros(((n + m) * q, (n + m) * q))
Q_norm[:n*q, :n*q] = Q_norm_u
Q_norm[n*q:, n*q:] = Q_norm_v

diag_s    = np.diag(S.T @ S)
Q_nonzero = np.diag(-lambda_nonzero * diag_s)

Q = Q_main + Q_norm + Q_nonzero

Q_min, Q_max = np.min(Q), np.max(Q)
Q_scaled  = ((Q - Q_min) / (Q_max - Q_min)) * 255 - 128
Q_clipped = np.clip(np.round(Q_scaled), -128, 127)
Q_qubo    = kw.qubo.adjust_qubo_matrix_precision(Q_clipped, bit_width=8)
print(f"Q_qubo shape: {Q_qubo.shape}")
print(f"Q_qubo range: [{np.min(Q_qubo)}, {np.max(Q_qubo)}]")

In [ ]:
# Convert to Ising model (kaiwu 1.3.1)
ising_mat, ising_bias = kw.conversion.qubo_matrix_to_ising_matrix(Q_qubo)
n_vars = ising_mat.shape[0]
variables = [f"x[{i}]" for i in range(n_vars)]
ising_model = kw.ising.IsingModel(
    variables=variables,
    ising_matrix=ising_mat,
    bias=ising_bias
)
print(f"Ising matrix shape: {ising_mat.shape}")

In [ ]:
# Submit to CIM (solve #1: submit)
start_time_quantum = time.time()

optimizer = kw.cim.CIMOptimizer(
    task_name='electricity_recommendation',
    task_mode='quota'
)
optimizer.solve(ising_model.get_matrix())
print("Task submitted, waiting for CIM to finish...")

In [ ]:
# Retrieve results (solve #2: fetch)
# Run this cell after CIM finishes
solution_ising = optimizer.solve(ising_model.get_matrix())

end_time_quantum = time.time()
time_quantum = end_time_quantum - start_time_quantum
print(f"Quantum solve time: {time_quantum:.4f} s")
print(f"Solution shape: {solution_ising.shape}")

In [ ]:
# Decode quantum results -> rank-1 reconstruction
solutions        = solution_ising[:, :-1]
deltas           = solution_ising[:, -1]
solutions_binary = (solutions * deltas[:, np.newaxis] + 1) / 2

energies    = [sig @ Q_qubo @ sig for sig in solutions_binary]
best_idx    = np.argmin(energies)
best_binary = solutions_binary[best_idx, :]
print(f"Best solution index: {best_idx}, energy = {energies[best_idx]:.6f}")

w     = S @ best_binary
u_raw = w[:n]
v_raw = w[n:]

u_q = u_raw / (np.linalg.norm(u_raw) + 1e-12)
v_q = v_raw / (np.linalg.norm(v_raw) + 1e-12)

sigma_q      = u_q.T @ R @ v_q
R_pred_q     = np.clip(sigma_q * np.outer(u_q, v_q), 0, 1)
rmse_q       = np.sqrt(np.mean((R_pred_q[mask] - R_full[mask])**2))

print(f"Quantum SVD singular value σ = {sigma_q:.6f}")
print(f"Quantum rank-1 RMSE = {rmse_q:.4f}")

In [ ]:
# Evaluation
print("=" * 80)
print("Electricity Load - Recommendation Results (Power System)")
print("=" * 80)
print(f"Classical SVD (k=3):")
print(f"  RMSE:  {rmse_list[2]:.4f}")
print(f"  Time:  {time_manual:.4f} s")
print(f"Quantum SVD (rank-1):")
print(f"  RMSE:  {rmse_q:.4f}")
print(f"  Time:  {time_quantum:.4f} s")
print("=" * 80)

In [ ]:
# Results summary table
print("Recommendation System Results Summary")
print("=" * 60)
print(f"{'Method':<20} {'RMSE':<15} {'Time(s)':<15}")
print("-" * 60)
for k, rmse in zip(k_list, rmse_list):
    print(f"{'Classical SVD k='+str(k):<20} {rmse:<15.4f} {time_manual:<15.4f}")
print(f"{'Quantum SVD k=1':<20} {rmse_q:<15.4f} {time_quantum:<15.4f}")
print("=" * 60)

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
fig.suptitle("SVD Recommendation System (Power System - Electricity Load)", fontsize=14, fontweight="bold")

# Observed matrix
im0 = axes[0].imshow(R_obs, cmap="RdYlGn", aspect="auto", vmin=0, vmax=1)
axes[0].set_title("Observed Electricity Matrix(white=missing)", fontsize=10)
axes[0].set_xlabel("Time Slot")
axes[0].set_ylabel("User")
plt.colorbar(im0, ax=axes[0])

# Classical rank-3 reconstruction
im1 = axes[1].imshow(R_pred_best, cmap="RdYlGn", aspect="auto", vmin=0, vmax=1)
axes[1].set_title(f"Classical SVD (k=3)RMSE={rmse_list[2]:.3f}  Time={time_manual:.3f}s", fontsize=10)
axes[1].set_xlabel("Time Slot")
axes[1].set_ylabel("User")
plt.colorbar(im1, ax=axes[1])

# Quantum rank-1 reconstruction
im2 = axes[2].imshow(R_pred_q, cmap="RdYlGn", aspect="auto", vmin=0, vmax=1)
axes[2].set_title(f"Quantum SVD (k=1)RMSE={rmse_q:.3f}  Time={time_quantum:.3f}s", fontsize=10)
axes[2].set_xlabel("Time Slot")
axes[2].set_ylabel("User")
plt.colorbar(im2, ax=axes[2])

# RMSE comparison
all_methods = [f"k={k}" for k in k_list] + ["Quantum k=1"]
all_rmses   = rmse_list + [rmse_q]
bar_colors  = ["#3498DB"] * len(k_list) + ["#E74C3C"]
bars = axes[3].bar(all_methods, all_rmses, color=bar_colors, alpha=0.85, edgecolor="white")
for bar, r in zip(bars, all_rmses):
    axes[3].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                 f"{r:.3f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
axes[3].set_ylabel("RMSE")
axes[3].set_title("RMSE Comparison", fontsize=10)
axes[3].grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("results/algo7_recommendation_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/algo7_recommendation_comparison.png")